# 09 — Robustness

This notebook tests the sensitivity of the Phase 1 risk-based methods to the covariance lookback window, under the design pre-registered in §1, on the notebook 08 runs and harness.

## §0 Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import maplab as ml
import _nb08_helpers as h8
import _nb09_helpers as h
from maplab import Panel, apply_style

apply_style()
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)
MKT_simple = (simple_returns["SPY"] - rf_daily).dropna()

F4 = lambda v: f"{v:.4f}"
E3 = lambda v: f"{v:.3e}"

print(f"prices: {prices.shape[0]:,} days x {prices.shape[1]} assets "
      f"({prices.index.min().date()} -> {prices.index.max().date()})")
print(f"train/test split: {split_ts.date()}")

rdates = ml.rebalance_dates(log_returns.index)
rdates = rdates[rdates >= ml.first_eligible_rebalance(log_returns.index)]
assert len(rdates) == 208, f"expected 208 rebalance labels, got {len(rdates)}"
print(f"rebalance labels: n={len(rdates)}  first={rdates[0].date()}  last={rdates[-1].date()}")

## §1 Pre-registration

In [ ]:
reg = ml.inference.load_registration("nb09")
ml.inference.render_registration(reg)

## §2 Core runs and reproduction gate

Backtests the 8 notebook 08 core runs at 252 days from the default start and checks their Sharpe ratios against the Phase 1 values.

In [ ]:
CORE = h.CORE
core = h.build_core_runs(simple_returns, panel)
for name, r in core.items():
    assert r["wlog"].index.equals(rdates), f"{name}: wlog index != rdates"
print(f"{len(core)} core runs completed: {', '.join(core)}")

core_summary = h8.summary_table(core, CORE, split_ts, rf_daily)
ml.inference.check_reproduction(core_summary, {m: h8.REPRO_KEYS[m] for m in CORE})
print("GATES PASS")

## §3 Lookback grid

Backtests GMV, MDP, ERC and HRP (sample Σ) at 252, 504 and 756 days from a common start, with a weight cross-check against the §2 runs and the summary table in the three windows.

In [ ]:
common_start = h.common_start(log_returns.index, 756)
cs_dates = ml.rebalance_dates(simple_returns.index)
cs_dates = cs_dates[cs_dates >= common_start]
print(f"common start: {common_start.date()}  rebalances: {len(cs_dates)}  in test: {int((cs_dates > split_ts).sum())}")
assert common_start == pd.Timestamp("2011-01-31")
assert len(cs_dates) == 184 and int((cs_dates > split_ts).sum()) == 40

GRID = h.GRID
grid = h.build_grid_runs(simple_returns, panel, common_start)
for name, r in grid.items():
    assert r["wlog"].index.equals(cs_dates), f"{name}: wlog index != common-start rebalances"
    assert r["diag"]["n_rebalances"] == 184
print(f"{len(grid)} grid runs completed: {', '.join(grid)}")

Grid 252-day target weights against the §2 default-start target weights on the 184 common rebalance dates.

In [ ]:
dw = h.weight_crosscheck(grid, core)
for m, v in dw.items():
    print(f"{m}: max |dw| (grid 252d vs default start) = {E3(v)}")

print("Note: the 252d grid runs start at 2011-01-31, two years after the Phase 1 start, "
      "so their Sharpe ratios are not Phase 1 reproductions and differ from the Phase 1 tables by construction.")

Summary table for the 12 grid runs in the three windows of the common-start sample.

In [ ]:
grid_summary = h8.summary_table(grid, GRID, split_ts, rf_daily)

Xg = h8.excess_frame(grid, GRID, rf_daily)
XgW = h8.window_frames(Xg, split_ts)
Xg_full, Xg_test = XgW["full"], XgW["test (> split)"]
seg0 = simple_returns.index[simple_returns.index >= common_start][0]
assert Xg_full.index[0] == seg0
for w, Xw in XgW.items():
    print(f"{w:<18} T={len(Xw):>5}  first={Xw.index[0].date()}  last={Xw.index[-1].date()}  "
          f"NW lag={ml.robust.nw_lag(len(Xw))}")
print()
print(grid_summary.to_string(float_format=F4))

## §4 H3 — lookback (registered)

Delta-method Newey–West test of each method's 504-day and 756-day Sharpe ratio against its 252-day Sharpe ratio on the common-start full window, Holm-adjusted over the 8 tests, with the test-window difference.

In [ ]:
T_full, T_test = len(Xg_full), len(Xg_test)
rows = []
for m in h.GRID_METHODS:
    base = h.grid_label(m, 252)
    for L in (504, 756):
        lab = h.grid_label(m, L)
        r = ml.robust.sharpe_diff_hac(Xg_full[lab], Xg_full[base])
        assert r["L"] == ml.robust.nw_lag(T_full)
        rt = ml.robust.sharpe_diff_hac(Xg_test[lab], Xg_test[base])
        assert rt["L"] == ml.robust.nw_lag(T_test)
        rows.append({"id": f"H3_{m}_{L}", "method": m, "lookback": L, "dSR_full": r["d_sr"], "se": r["se"],
                     "t": r["t"], "p_raw": r["p"], "dSR_test": rt["d_sr"],
                     "same_sign_test": bool(np.sign(rt["d_sr"]) == np.sign(r["d_sr"]))})

h3 = pd.DataFrame(rows).set_index("id")
h3["p_holm"] = ml.robust.holm(h3["p_raw"].to_numpy())
h3_table = h3[["dSR_full", "se", "t", "p_raw", "p_holm", "dSR_test", "same_sign_test"]]
print(h3_table.to_string(float_format=F4))

h3_stats = {}
for hid, row in h3.iterrows():
    h3_stats[hid] = dict(
        display=f"ΔSR {row['dSR_full']:+.2f}, t {row['t']:.2f}, Holm p {row['p_holm']:.3f}",
        p_adj=float(row["p_holm"]), same_sign_test=bool(row["same_sign_test"]),
    )
reg_ids = {x["id"] for x in reg["hypothesis"]}
assert len(h3_stats) == 8 and set(h3_stats) <= reg_ids
assert set(h3_stats) == {x["id"] for x in reg["hypothesis"] if x["id"].startswith("H3_") and x["kind"] == "adjusted_null"}

H3 descriptive: annual turnover, median effN, median UUP target weight and UUP target weight at 2022-12-31 per method and lookback.

In [ ]:
h3_desc = h.h3_desc_table(grid, grid_summary, "2022-12-31")
print(h3_desc.to_string(float_format=F4))

turn_cmp = pd.DataFrame({
    "grid 252d (common start)": [grid_summary.loc[(h.grid_label(m, 252), "full"), "ann_turnover"] for m in h.GRID_METHODS],
    "Phase 1 (default start)": [core_summary.loc[(h.P1_LABEL[m], "full"), "ann_turnover"] for m in h.GRID_METHODS],
}, index=h.GRID_METHODS)
print()
print("ann_turnover, full window:")
print(turn_cmp.to_string(float_format=F4))

### Figure F6 — full-window Sharpe, annual turnover and median effN by lookback, one line per method.

In [ ]:
sharpe_full = grid_summary.xs("full", level="window")["sharpe"]
fig = h.lookback_panels(h3_desc, sharpe_full)
plt.show()

### Figure F7 — UUP target weight at each rebalance by lookback, one panel per method; the dotted line marks the train/test split.

In [ ]:
fig = h.uup_paths(grid, split_ts)
plt.show()

## §5 H4 — forecast bias (registered)

Realized over ex-ante volatility per holding segment for the 8 core runs, with the notebook 07 segment construction and a reproduction gate against its printed means of q.

In [ ]:
import _nb07_helpers as h7

q_table, n_dropped, _ = h7.forecast_bias_table(panel, rdates, core, simple_returns, CORE)
print(f"segments dropped (< 10 remaining days after excluding cost-booking day): {n_dropped}")
n_seg = q_table.groupby("method").size().reindex(CORE)
assert n_seg.nunique() == 1, n_seg.to_dict()
print(f"segments per method: {int(n_seg.iloc[0])}")

Q_PHASE1 = {"HRP(S)": -0.0999, "GMV(S)": -0.0356, "MDP(S)": -0.0518, "ERC(S)": -0.1049, "EW": -0.1885}
q_mean = q_table.groupby("method")["q"].mean()
for m, exp in Q_PHASE1.items():
    got = round(float(q_mean[m]), 4)
    assert got == exp, f"H4 GATE FAILED: {m} mean q {got} != {exp}"
print("H4 GATE PASS")

Mean variance ratio per method with its segment block-bootstrap standard error and interval, on the full and test windows.

In [ ]:
B4, BLOCK4, SEED4 = 10000, 12, 20260923
h4_rows, dec_rows = [], []
for m in CORE:
    seg = q_table[q_table["method"] == m].sort_values("asof")
    test = seg[seg["asof"] > split_ts]
    d = ml.robust.forecast_bias_decomposition(seg["realized_vol"], seg["exante_vol"])
    dt = ml.robust.forecast_bias_decomposition(test["realized_vol"], test["exante_vol"])
    assert abs(d["q_bar"] - float(q_mean[m])) <= 1e-12
    v_full = ((seg["realized_vol"] / seg["exante_vol"]) ** 2).to_numpy()
    draws = ml.robust.segment_block_bootstrap_mean(v_full, block=BLOCK4, B=B4, seed=SEED4)
    sd = float(draws.std(ddof=1))
    lo, hi = np.percentile(draws, [2.5, 97.5])
    h4_rows.append({"method": m, "vbar_full": d["vbar"], "sd": sd, "t": (d["vbar"] - 1) / sd,
                    "lo": float(lo), "hi": float(hi), "vbar_test": dt["vbar"],
                    "n_full": len(seg), "n_test": len(test)})
    for w, dd in (("full", d), ("test", dt)):
        dec_rows.append({"method": m, "window": w, "q_bar": dd["q_bar"],
                         "half_log_vbar": dd["half_log_vbar"], "J": dd["J"]})

h4 = pd.DataFrame(h4_rows).set_index("method")
print(h4.to_string(float_format=F4))

H4 descriptive: mean log ratio, half log mean variance ratio and Jensen gap per method, full and test windows.

In [ ]:
h4_dec = pd.DataFrame(dec_rows).set_index(["method", "window"])
print(h4_dec.to_string(float_format=F4))

H4_KEYS = {"EW": "EW", "GMV(S)": "GMV", "MaxSharpe(S)": "MaxSharpe", "BMV(0.3)": "BMV",
           "BL(0.1,S)": "BL", "MDP(S)": "MDP", "ERC(S)": "ERC", "HRP(S)": "HRP"}
assert list(H4_KEYS) == CORE
h4_stats = {}
for m, key in H4_KEYS.items():
    v, t, vt = float(h4.loc[m, "vbar_full"]), float(h4.loc[m, "t"]), float(h4.loc[m, "vbar_test"])
    h4_stats[f"H4_{key}"] = dict(display=f"vbar {v:.3f}, t {t:.2f}, test vbar {vt:.3f}",
                                 t_full=t, mean_test=vt - 1)
assert set(h4_stats) <= reg_ids
assert set(h4_stats) == {x["id"] for x in reg["hypothesis"] if x["id"].startswith("H4_") and x["kind"] == "directional_t"}

### Figure F8a — full-window mean variance ratio with the segment block-bootstrap 95% interval; the dashed line marks 1.

In [ ]:
f8a = h4.reset_index()[["method", "vbar_full", "lo", "hi"]]
fig = ml.plotting.forest_plot(f8a, "method", "vbar_full", "lo", "hi", ref=1.0,
                              xlabel="mean realized/ex-ante variance ratio (full window)")
plt.show()

### Figure F8b — full-window decomposition of the mean log ratio into half log mean variance ratio and Jensen gap.

In [ ]:
fig = h.bias_decomposition_bars(h4_dec.xs("full", level="window").loc[CORE])
plt.show()

## §6 H5 — train vs test Sharpe ranks (descriptive)

Train and test Sharpe ratios of the 8 core runs from the §2 summary table, their ranks and the Spearman rank correlation.

In [ ]:
h5 = pd.DataFrame({
    "SR_train": core_summary.xs("train (<= split)", level="window").loc[CORE, "sharpe"],
    "SR_test": core_summary.xs("test (> split)", level="window").loc[CORE, "sharpe"],
})
h5["rank_train"] = h5["SR_train"].rank(ascending=False)
h5["rank_test"] = h5["SR_test"].rank(ascending=False)
print(h5.to_string(float_format=F4))
rho_s = float(h5["SR_train"].corr(h5["SR_test"], method="spearman"))
print(f"\nSpearman rank correlation, train vs test Sharpe (8 core runs): {rho_s:.4f}")

### Figure F9 — annualized volatility and return of each core run, train point to test point.

In [ ]:
fig = h.train_test_arrows(core_summary, CORE)
plt.show()

### §6b Variance-ratio profile (descriptive, not registered — added after the H4 results)

The typical segment and the largest segments of v = (realized / ex-ante)² per core method, and v̄ without the segment starting 2020-02-29.

In [ ]:
vr = h.variance_ratio_profile(q_table, CORE)
print(vr.to_string(float_format=lambda v: f"{v:.4f}"))

## §7 Verdict table

Registered rules from §1 applied mechanically to the H3 and H4 statistics.

In [ ]:
table = ml.inference.evaluate(reg, {**h3_stats, **h4_stats})
assert len(table) == 16
print(table.to_string(index=False))
print()
print(table["verdict"].value_counts().to_string())

## TAKEAWAY — the 8 lookback nulls hold; the forecast-bias claim is not supported, and the variance ratio points the other way

**H3 (lookback): the null holds for all 8 tests.** From the common 2011 start, every 504-day and 756-day run has a higher full-window Sharpe ratio than its 252-day run (ΔSR +0.06 to +0.10), and all 8 keep that sign in the test window. None survives the Holm adjustment: the smallest adjusted p is 0.338 (ERC 504d, raw t 2.03). The 8 differences are not independent — the 504-day and 756-day runs of a method share most of their weights — so the common sign is weak evidence. Descriptively, a 756-day window cuts annual turnover roughly in half or more (GMV 0.71 to 0.29, HRP 0.62 to 0.28) and lowers the UUP weight at 2022-12-31 (GMV 58.1% to 46.7%, ERC 48.4% to 38.5%), while the median UUP weight moves by at most 0.03. A longer window delays the UUP corner; it does not remove it.

**H4 (forecast bias): not supported for any of the 8 core methods, and the evidence runs the other way.** The registered claim was v̄ < 1. Every method has v̄ > 1, from 1.30 (EW) to 1.73 (MaxSharpe), and only EW has a test-window v̄ below 1 (0.90). MaxSharpe is the only method whose t exceeds 2 in absolute value, and its sign (+2.34) is the opposite of the claim. The negative mean log ratio seen in notebook 07 comes entirely from the Jensen gap J (0.20 to 0.37).

**What the variance ratio shows (descriptive, §6b).** The 252-day ex-ante vol over-forecasts the typical month and badly under-forecasts a few stress months. Median v is below 1 for every method (0.62 to 0.97), and 53% to 70% of segments have v below 1. The segment starting 2020-02-29 is the largest for all 8 methods (v from 37 for MDP to 74 for HRP), and the segment starting 2025-03-31 is second for 7 of 8. The five largest segments carry 23% to 38% of the sum of v, and removing the 2020 segment alone brings v̄ to 0.98 to 1.42. The mean log ratio follows the typical month, v̄ follows the tail, and J is the distance between them.

**Consequence for notebook 07 H7.** H7 stays SUPPORTED on its registered statistic, but its reading changes. The HRP − GMV difference in mean q of −0.064 splits into +0.010 from ½·log v̄ and −0.074 from J. HRP and GMV differ in how their forecast errors are distributed, not in the average level of their variance forecast; in variance terms HRP is marginally more under-forecast (v̄ 1.53 against 1.50).

**H5 (descriptive).** The Spearman rank correlation between train and test Sharpe across the 8 core runs is 0.50. MaxSharpe moves from 6th to 1st, BL(0.1) from 5th to 8th, and MDP stays 3rd.

**Reading.** Lookback length moves turnover and the timing of the UUP corner far more than it moves any measurable performance. A trailing-covariance vol forecast is wrong in both directions at once — too high most of the time, far too low in stress — which is the pattern a volatility-managed overlay is meant to address in Phase 3.